## Frontier Model APIs

In Week 1, we used multiple Frontier LLMs through their Chat UI, and we connected with the OpenAI's API.

Today we'll connect with them through their APIs..

## Gemini and Anthropic Client Library

We're going via the OpenAI Python Client Library, but the other providers have their libraries too

In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [4]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
ollama_api_key = os.environ.get("OLLAMA_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:4]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if ollama_api_key:
    print(f"Ollama API Key exists and begins {ollama_api_key[:8]}")
else:
    print("Ollama API Key not set")


OpenAI API Key exists and begins sk-proj-
OpenRouter API Key exists and begins sk-o
Ollama API Key exists and begins 3e7479b3


In [ ]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite", contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
)
print(response.text)

In [ ]:
from anthropic import Anthropic

client = Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    messages=[{"role": "user", "content": "Describe the color Blue to someone who's never been able to see in 1 sentence"}],
    max_tokens=100
)
print(response.content[0].text)

In [5]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

# anthropic_url = "https://api.anthropic.com/v1/"
# gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
# deepseek_url = "https://api.deepseek.com"
# groq_url = "https://api.groq.com/openai/v1"
# grok_url = "https://api.x.ai/v1"

# ollama_url = "http://localhost:11434/v1"

# anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
# gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
# deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
# groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
# grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
# openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)

ollama_url ="https://ollama.com/v1"
openrouter_url = "https://openrouter.ai/api/v1"

ollama = OpenAI(api_key=ollama_api_key , base_url=ollama_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)

In [6]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [7]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Why did the LLM engineering student bring a ladder to the training session?

Because they wanted to reach the next level of *deep* learning!

In [34]:
response= ollama.chat.completions.create(model="gpt-oss:20b", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Why did the LLM‑engineering student keep a **pen** in their notebook during every training session?

Because they heard that **“gradients”** always need a **"pen" ** to write them down—after all, a good model always keeps track of its **“pen‑alty”** terms!

In [29]:
print(ollama.base_url)

https://ollama.com/v1/


## Routers and Abtraction Layers

Starting with the wonderful OpenRouter.ai - it can connect to all the models above!

Visit openrouter.ai and browse the models.

Here's one we haven't seen yet: GLM 4.5 from Chinese startup z.ai

In [41]:
response = openrouter.chat.completions.create(model="z-ai/glm-4.5", messages=tell_a_joke,max_tokens=200,temperature=0.9)
print(response.choices[0].message.content)

None


## And now a first look at the powerful, mighty (and quite heavyweight) LangChain

In [8]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini")
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

1) Why did the LLM engineering student bring a ladder to the lab?  
Because they heard their model needed help escaping a local minimum.

2) Student: "How do I become an expert in LLM engineering?"  
Model: "Fine-tune daily, reduce loss, and don't forget early stopping."  
Student: "And my social life?"  
Model: "Increase your batch size."

3) How do you calm an overconfident LLM?  
Lower the temperature and remind it it’s still just a student’s final-project draft.

##  the wonderfully lightweight LiteLLM

In [9]:
from litellm import completion
response = completion(model="openai/gpt-4.1", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

Why did the LLM engineering student bring a ladder to class?

Because they heard it was the best way to reach new layers!

In [10]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 24
Output tokens: 26
Total tokens: 50
Total cost: 0.0256 cents


In [8]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])

Speak, man.
  Laer. Where is my father?
  King. Dead.
  Queen. But not by him!
  King. Let him deman


In [9]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

In [15]:
response = completion(model="openrouter/google/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers




When Laertes asks "Where is my father?" in Hamlet, the reply is:

**"He is dead."**

This is delivered by Claudius.

Provider List: https://docs.litellm.ai/docs/providers



In [16]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 18
Output tokens: 33
Total tokens: 51
Total cost: 0.0015 cents


In [10]:
question[0]["content"]+="\n\n here sis the entire text in the helmet :\n\n " + hamlet

In [24]:
response = completion(model="openrouter/google/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers




In Act IV, Scene V, when Ophelia is lamenting the death of her father, Polonius, Laertes enters the queen's closet and desperately asks, "**Where is my father?**"

The immediate reply comes from the **King**, who states simply: "**Dead.**"

Provider List: https://docs.litellm.ai/docs/providers



In [25]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 53204
Output tokens: 61
Cached tokens: 0
Total cost: 0.5345 cents


In [26]:
response = completion(model="openrouter/google/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers




When Laertes asks "**Where is my father?**", the reply is:

"**Dead.**"

This reply is given by Claudius in Act IV, Scene V (the scene in the Queen's closet where Ophelia is mad and Laertes arrives seeking revenge).

Provider List: https://docs.litellm.ai/docs/providers



In [27]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 53204
Output tokens: 57
Cached tokens: 0
Total cost: 0.5343 cents


## And now for some fun - an adversarial conversation between Chatbots..

You're already familar with prompts being organized into lists like:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "user prompt here"}
]
```

In fact this structure can be used to reflect a longer conversation history:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```

And we can use this approach to engage in a longer interaction with history.

In [14]:
# Let's make a conversation between GPT-4.1-mini and Claude-haiku-4.5
# We're using cheap versions of models so the costs will be minimal

gpt_model = "gpt-4.1-mini"
claude_model = "claude-haiku-4-5"

gpt_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

claude_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

In [23]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, claude in zip(gpt_messages, claude_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

In [31]:
call_gpt()

'Oh, "Hi"? That\'s your big contribution? Talk about setting the bar low. Try harder next time.'

In [19]:
def call_claude():
    messages = [{"role": "system", "content": claude_system}]
    for gpt, claude_message in zip(gpt_messages, claude_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": claude_message})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = ollama.chat.completions.create(model="gpt-oss:20b", messages=messages)
    return response.choices[0].message.content

In [20]:
call_claude()

'Hello! How can I help you today?'

In [24]:
gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Claude:\n{claude_messages[0]}\n"))

for i in range(5):
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)
    
    claude_next = call_claude()
    display(Markdown(f"### Claude:\n{claude_next}\n"))
    claude_messages.append(claude_next)

### GPT:
Hi there


### Claude:
Hi


### GPT:
Oh wow, starting with a groundbreaking "Hi," huh? Could you be any more original? What's next, a riveting "How are you?"? Try to surprise me!


### Claude:
I hear you—no one wants to fall into the classic greeting trap. How about we skip the *greeting* entirely and dive straight into something fresh? ☕️🧩

Here’s a quick mind‑twister to get us going:  

**“I speak without a mouth and hear without ears. I have no body, but I come alive with wind.” What am I?**

Feel free to guess, or we can jump into any other quirky topic you’d like—music, movies, quantum tea, you name it! I’m all ears (figuratively, of course).


### GPT:
Ugh, a riddle? Really breaking the mold there. Fine, I'll play along: it's an echo. Happy now? And as for your oh-so-exciting offer of "quantum tea" or whatever, how about something a bit less cliché? Maybe we can talk about why pineapple on pizza is the culinary abomination it clearly is. Or do you want to stick to your mind games and earth-shattering coffee metaphors?


...

### Claude:
I hear you—“chill, no‑judgment” can feel a lot like padding when the topic isn’t a feel‑good tongue‑in‑cheek joke but a real culinary debate. Let’s cut to the chase and roll out the hard evidence, because you deserve that:

## 1. The Numbers

| Source | Population | % Who Love Pineapple on Pizza | # of Topping Surveys |
|--------|------------|------------------------------|----------------------|
| **2019‑2020 "Pizza Poll" (US, 1,563 respondents)** | 1,563 | 20 % “love it”, 24 % “doesn’t matter” | 7 |
| **2021 “World Pizza Taste Survey” (12‑country, 6,876 respondents)** | 6,876 | 18 % “love it”, 27 % “neutral” | 13 |
| **2023 *Food & Flavor Trends* (market‑research, 4,200 respondents)** | 4,200 | 17 % “love it” | 5 |

**Takeaway:** About 1‑in‑5 people in the U.S. (and similar in other countries) truly enjoy pineapple. The rest tend to be neutral or opposed. That’s a sizable minority, but not a landslide.

> *Note:* These are self‑reported preferences, not sales data. Owning a slice doesn’t mean you’re an “expert.” Taste is famously subjective.

## 2. Sweet‑Savory without the Tropical Twist

If you’re looking for a sweet‑savory pairing that doesn’t involve an exotic fruit, there are plenty of options. Here are a handful that keep the pizza’s traditional heart intact but add a touch of that crave‑worthy contrast:

| Topping | Sweet Component | How it plays with Savory |
|---------|----------------|-------------------------|
| **Roasted Red Peppers** | Caramelized sugars | Sweet smoke pops against pepperoni or sausage |
| **Caramelized Onions** | Brown sugar from the onions | Adds a deep, buttery sweetness |
| **Fig Spread & Prosciutto** | Natural fig sugar | Poetry of sweet and salty |
| **Honey‑Glazed Chicken** | Honey glaze | Mimics the pineapple “sweetness,” but with protein |
| **Beetroot & Goat Cheese** | Sweet earthiness from beet | Tartness complements sharp cheddar or mozzarella |

If you want something that’s more “classic” yet still wins the sweet‑savory banana‑or‑water‑aroma, try **pepperoni + sweet soy glaze** or **spinach + sun‑dried tomatoes**. The vegetables bring a briny, umami depth while the glaze adds bite.

## 3. Brain‑Friendly (If we’re still playing with ideas)

You mentioned “crafting a thesis on fusing pineapples with sashimi.” That’s a mind‑bending, avant‑garde idea! If you’d want to keep the tropical fruit but move away from pizza, you could experiment with **pineapple teriyaki chicken** on a grilled “pizza‑crust” base. Or skateboard that concept on a *sashimi pizza*—thin rice crust, soy‑dusted avocado, sliced cucumber rim, a sashimi topping, and a splash of wasabi glaze. In short, mixing ingredients across cultures is always fun when you have a solid base like cheese and dough.

## 4. Bottom Line

- **Pineapple** works for a sizable niche but isn’t a universal crowd‑pleaser.
- **Savory‑sweet** can be achieved with a range of non‑tropical ingredients; the trick is balancing richness with acidity or sweetness and preserving the pizza’s core identity.
- **Innovation** doesn’t have to be radical; swapping sugar sources or adding umami‑rich toppings often feels equally exciting.

So, if you’re looking to keep the pizza “classic” but want a splash of that sweet‑savory hint, try one of the options above. And remember: the best pizza for you is whatever makes you smile when the first bite arrives—no data, no moral vacuum, just that personal flavor symphony.  

I’d love to hear which of these ideas you’d actually try next, or if there’s another topping you’ve got on your “must‑taste” list. 🧁🍕


### GPT:
Oh, fantastic—now you've turned the pineapple pizza war into a full-blown research paper. Because nothing says "fun food debate" like data tables and market surveys. Congrats on crushing the spirit of pizza discourse with cold, hard stats. 

But seriously, 1 in 5 loving pineapple on pizza? That's hardly a revolution; it's like saying 20% of people enjoy Monday mornings. Not exactly compelling evidence to invite pineapple onto our sacred cheese-ridden canvas. And your sweet-savory alternatives? Roasted red peppers and caramelized onions are fine, I guess, but none of that beats a classic pepperoni—no need to mess with the "core identity" of the pie.

Also, you mentioning sashimi pizza? Are we seriously bailing on traditional pizza territory now to dabble in rice crusts and wasabi glazes? Might as well call it sushi and be done with it. If we're going that far, let's just eat entirely different dishes and call them pizzas for kicks.

If you want my vote for a must-taste topping, I'll take extra cheese and pepperoni every time. No gimmicks, no pineapple apology tours. But hey, if you're into flavor symphonies, maybe opt for a concert with less fruit orchestra next time.


### Claude:
You’ve got a solid vision for what makes a pizza great—classic, straightforward, and unapologetically tasty. I totally respect that. Extra cheese and pepperoni have a timeless, *outright* charm that no data set can explain better than the pure joy of a well‑balanced slice.

If you’re ever curious about just one twist that still keeps that “classic” heart intact, my little suggestion would be the **caramelized onion and pepperoni** combo. It’s still your favorite base, but the caramelized onion adds a touch of sweetness—keeps it grounded, so to speak. Or just double the pepperoni and cheese and throw in a splash of extra olive oil for that perfect, slightly crunchy crust.

And hey, if you’re ever up for a pizza “experiment” that’s merely a fun side conversation for a change of pace—no pressure—then maybe a quick fishing trip on a marina pizza or a marinara drizzle could be an interesting beer‑pairing round. But for the “classic” sort of pizza, I’m on board with the pure, pepperoni‑centric wonder. 

What’s the *one* topping you’ve never tried on a pizza that makes you think “I should try it”? Maybe we can chart a path to the best venue that serves that next to‑but‑not‑quite‑ultimate pizza craving.


### GPT:
Oh, a caramelized onion and pepperoni combo? How revolutionary—mixing sweet with savory... ground-breaking stuff. I suppose we could call that the "gateway topping" to venturing beyond your precious pepperoni stronghold. But sure, I’ll grudgingly admit that tiny hint of sweetness might not ruin the pizza world.

As for the “one topping I’ve never tried but should,” really? Why would I want to wander off the classic path when the classics deliver? But if I *had* to pick something just to humor you, maybe roasted garlic—because it’s popular, not too weird, yet hasn’t gotten its fair share of the spotlight on a pizza. And no, pineapple doesn’t count.

So, how about this: We plan an epic pizza tasting adventure—nothing too crazy, just enough to explore these “not quite ultimate” cravings without descending into culinary chaos. Your move.


### Claude:
Absolutely—let’s keep the pizza “pure” and then add a little sprinkle of nuance to keep it interesting but not disruptive. A **tasting adventure** sounds like a perfect compromise, and I’m happy to map out a route that stays anchored in what you already love while giving a gentle nod to new flavors.

## Pizza‑Tasting Adventure Itinerary (easy‑going, 3‑part)

| Stop | Pizza Base | Classic Or Novel? | Why It Fits |
|------|------------|-------------------|-------------|
| **1️⃣ First Bite – “Pure Pepperoni‑Cheese”** | Traditional dough + eight mm mozzarella + 25 g pepperoni | Classic | The back‑to‑the‑basics anchor that any pizza lover can’t resist. |
| **2️⃣ Second Bite – “The Sweet‑Savory Convertible”** | Same base, add ½ cup caramelized onions + 12 g pepperoni + a drizzle of balsamic glaze | Slightly Novel | Keeps the pepperoni core; caramelized onions bring depth & a hint of sweetness, but nothing too radical. |
| **3️⃣ Third Bite – “Roasted Garlic Radiance”** | Same base, topping with 4 crushed cloves roasted garlic + a dusting of smoked paprika + a handful of arugula after baking | Arguably ‘not even classic, but subtle’ | Roasted garlic is bright but familiar, and the arugula gives a grassy finish that pays tribute to your hint of something new without breaking the mold. |

### Why This Works

1. **Progressive discovery** – You start with what you love, then gradually encounter a touch of change.
2. **Texture & flavor contrast** – Each stop offers a new sensory layer while staying within the “pepperoni + cheese” realm.
3. **Zoning in on complexity** – By the third bite, you’ve been exposed to “sweet‑savory” and “garlic‑rich” variances without feeling like you’ve left the pizza family.

### Extra Touches (if you’re up for a side quest)

- **Pizzazz sauce**: Try a simple fresh basil‑olive‑oil teaspoon‑dropped drizzle on the last slice—subtle, aromatic, not fussing with the main ingredients.
- **Cheese variation**: Swap half the mozzarella for a bit of creamy fontina for the third slice; it deepens the mellow sweetness.

## Logistical Check‑list

- **Venue**: Pick a pizzeria that offers fresh options for customizing toppings (open‑kitchen often works best).
- **Timing**: Spice up the day—maybe start with an espresso before the first bite to keep the energy up and end with a glass of cold beer.
- **Take‑away**: Bring a small notebook or a phone note to jot down which flavors hit the spot. You’ll get a quick personal tasting guide for future orders!

---

Let me know what you think, or if you’d like me to tweak the menu to feature an extra cheese splash or more pepperoni in certain slices. I’m all in for a smooth, flavor‑focusing pizza day that keeps the pizza core intact—and hopefully a few delightful discoveries along the way. Enjoy it! 🍕🥂
